# TeenPattiBench — fine-tune Llama 3.2 3B (LoRA, L4 GPU)

What this notebook does, in order:

1. **Splits** `teenpattibench_finetune.jsonl` into **80,000 train / 10,000 validation /
   10,000 test**, **by situation** — so no blind prompt leaks across splits through its 30
   identical copies, and every contrast group stays whole.
2. **Evaluates base Llama on this same GPU** on the internal test split, so the
   before/after comparison has no hardware difference in it.
3. **Trains a LoRA adapter**: loss on the answer tokens only, optional per-example
   confidence weights, contrast groups kept together, checkpoints to Google Drive.
4. **Checks validation every 250 steps** and keeps the best adapter by
   *balanced accuracy* (mean recall over the six actions), which penalises collapsing onto
   one answer.
5. **Tests the best adapter once** on the **10,000-row internal test split**.

**The 2,000-question benchmark eval is not used anywhere in this notebook** — not for
training, not for choosing checkpoints, not for testing. It is not even loaded. The
fine-tune file was generated disjoint from it, so it stays a clean, untouched test for a
separate evaluation run later. Checkpoints are chosen on validation alone; the internal
test is used only in steps 2 and 5.

### Before you run
1. **Runtime → Change runtime type → L4 GPU.**
2. You already have Llama access; add `HF_TOKEN` as a Colab secret (🔑) with notebook access.
3. **Put these three files in one place** — either `/content` (📁 Files panel) or a Drive
   folder named `teenpattibench_data` (recommended: the 182 MB training file then survives
   disconnects):
   `teenpattibench_finetune.jsonl` · `teenpattibench_system_prompt.txt` · `metrics.py`

   Do **not** upload the benchmark eval files; this notebook does not use them.
4. First time: set `MODE = "sanity"` in cell 1 and **Run all** (~10 min). It trains on 500
   items and checks it can learn them — it must reach ~100%. Then `MODE = "pilot"`
   (10,000 items), then `MODE = "full"`.
5. **Allow Google Drive** when cell 3 asks. If Colab disconnects, **Run all** again: it
   resumes from the last checkpoint and skips finished steps.

## 1. Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao"])

In [ ]:
MODE = "sanity"      # "sanity" (128 items, 80 steps) -> "pilot" (10k, 1 epoch) -> "full"

BASE_ID   = "meta-llama/Llama-3.2-3B-Instruct"
BASE_NAME = "Llama 3.2 3B"
TAG       = "llama3.2_3b"

# ---- LoRA and optimisation (see the fine-tuning recommendations) ---------------
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS   = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LR             = 2e-4
WARMUP_FRAC    = 0.03
WEIGHT_DECAY   = 0.0
MAX_GRAD_NORM  = 1.0
PER_DEVICE_BS  = 8
GRAD_ACCUM     = 8          # 8 x 8 = 64 examples per optimiser step
MAX_LEN        = 1024
# Saves GPU memory by recomputing activations instead of storing them (~25-35% slower).
# Needed on an L4 (22.5 GB). On an A100 (40 GB) set False for a faster run -- and you
# may also use PER_DEVICE_BS = 16, GRAD_ACCUM = 4 (still 64 per step). Results are the same.
GRAD_CHECKPOINTING = True
USE_WEIGHTS    = True       # the per-example `weight` field (label confidence, mean 1.0)
SEED           = 20260929

# ---- split of the 100,000-row fine-tune file (rows) --------------------------
SPLIT            = {"train": 80_000, "val": 10_000, "test": 10_000}
# Scoring all 10,000 validation items every check would take several minutes on an
# L4, so a label-balanced subset is checked during training and the full 10,000 once
# at the end. Every held-out blind prompt is always in the subset.
VAL_DURING_TRAIN = 2_000
VAL_EVERY        = 250       # optimiser steps
PLOT_EVERY       = 20        # live dashboard refresh (optimiser steps)
SAVE_EVERY       = 250

# ---- evaluation: identical to the batch eval notebook -------------------------
GEN_BATCH, MAX_NEW_TOKENS = 8, 16
RUN_BASELINE   = True        # base model on this GPU, before training (internal test)
RUN_FINAL_TEST = True        # best adapter, after training (internal test)
# Llama 3.2's chat template writes *today's* date into the system turn unless given one.
# Fixing it makes train, baseline and test prompts byte-identical across days.
DATE_STRING = "26 Jul 2024"

DRIVE_DATA_DIR = "/content/drive/MyDrive/teenpattibench_data"
OUT_DIR        = f"/content/drive/MyDrive/teenpattibench_finetune/{TAG}_{MODE}"

## 2. Packages and GPU

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "accelerate"])

import torch, transformers, peft, os, json, math, random, time, gc, collections, shutil
from packaging import version
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> L4.")
if not torch.cuda.is_bf16_supported():
    raise SystemExit("This GPU has no bfloat16. Use an L4 or A100.")
GPU = torch.cuda.get_device_name(0)
print(f"GPU {GPU} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
torch.manual_seed(SEED); random.seed(SEED)

## 3. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if MODE == "sanity" and os.path.isdir(OUT_DIR):
    shutil.rmtree(OUT_DIR)          # sanity is a throwaway test: always start clean
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
BEST_DIR = os.path.join(OUT_DIR, "best_adapter")
VAL_LOG  = os.path.join(OUT_DIR, "val_log.jsonl")
print("outputs ->", OUT_DIR)
print("existing:", sorted(os.listdir(OUT_DIR)) or "none (fresh run)")

## 4. Find and load the files

In [ ]:
# The benchmark eval (teenpattibench_eval*.jsonl) is deliberately NOT loaded.
NEED = ["teenpattibench_finetune.jsonl", "teenpattibench_system_prompt.txt", "metrics.py"]
def find(name):
    for d in ("/content", DRIVE_DATA_DIR):
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None
PATHS = {n: find(n) for n in NEED}
missing = [n for n, p in PATHS.items() if p is None]
if missing:
    raise SystemExit(f"Missing: {missing}\nPut them in /content or {DRIVE_DATA_DIR}, then re-run.")
for n, p in PATHS.items():
    print(f"  {n:<38} {p}")

sys.path.insert(0, os.path.dirname(PATHS["metrics.py"]))
import metrics
SYSTEM_PROMPT = open(PATHS["teenpattibench_system_prompt.txt"], encoding="utf-8").read()
ROWS = [json.loads(l) for l in open(PATHS["teenpattibench_finetune.jsonl"], encoding="utf-8")]
print(f"\nfine-tune rows {len(ROWS):,}")
if any(r["id"].startswith("eval-") for r in ROWS):
    raise SystemExit("benchmark eval items found in the fine-tune file -- wrong file?")
if len(ROWS) != 100_000:
    raise SystemExit("teenpattibench_finetune.jsonl should have 100,000 rows.")

## 5. Split: 80,000 train / 10,000 validation / 10,000 test — by situation

A *unit* is what must never be split: a **blind situation** (all 30 of its copies), a
**contrast group** (4–8 hands of one betting line), or a **single seen item**.

* **Blind:** 10% of situations to validation and 10% to test, per family × action
  (12 + 12 of the 128). Neither shares a blind situation with training. **But that is
  only 6 conversion situations per split (12 distinct prompts), far too few to measure
  conversion.** Conversion needs the benchmark eval (162 items), in a separate run after
  fine-tuning; read the internal splits for seen betting.
* **Contrast groups:** whole groups, ~10% of their rows to each held-out split.
* **Single seen items:** fill each held-out split to exactly 10,000 rows, in proportion to
  each action.

For scoring, identical prompts are counted once: a blind situation's 30 copies are the
same question (only the listed action order varies between some of them).

In [ ]:
def make_split(rows, seed):
    rng = random.Random(seed)
    units, stratum = collections.OrderedDict(), {}
    for i, r in enumerate(rows):
        m = r["meta"]
        if m["blind"]:
            u, st = ("blind", m["node"]), ("blind", m["family"], r["correct_action"])
        elif m.get("group_id"):
            u, st = ("group", m["group_id"]), ("group",)
        else:
            u, st = ("single", i), ("single", r["correct_action"])
        units.setdefault(u, []).append(i)
        stratum[u] = st
    by = collections.defaultdict(list)
    for u in units:
        by[stratum[u]].append(u)
    for st in by:
        by[st].sort(key=str); rng.shuffle(by[st])

    total = len(rows)
    frac = {k: SPLIT[k] / total for k in ("val", "test")}
    where, rows_in = {}, collections.Counter()
    def put(u, k):
        where[u] = k; rows_in[k] += len(units[u])

    # 1. blind situations: a fixed fraction of each family x action
    for st, us in by.items():
        if st[0] != "blind":
            continue
        k = {h: round(frac[h] * len(us)) for h in ("val", "test")}
        for j, u in enumerate(us):
            put(u, "test" if j < k["test"] else "val" if j < k["test"] + k["val"] else "train")
    # 2. contrast groups: whole groups up to ~10% of group rows each
    gs = by[("group",)]
    grows = sum(len(units[u]) for u in gs)
    tgt = {h: frac[h] * grows for h in ("val", "test")}
    got = collections.Counter()
    for u in gs:
        h = next((h for h in ("test", "val") if got[h] + len(units[u]) <= tgt[h]), "train")
        got[h] += len(units[u]) if h != "train" else 0
        put(u, h)
    # 3. single items: fill val and test to exactly SPLIT rows, proportional per action
    singles = {st: us for st, us in by.items() if st[0] == "single"}
    n_single = sum(len(us) for us in singles.values())
    for h in ("test", "val"):
        need = SPLIT[h] - rows_in[h]
        if need < 0:
            raise SystemExit(f"{h} already has {rows_in[h]} rows before singles")
        labs = sorted(singles, key=lambda st: -len(singles[st]))
        alloc = {st: int(need * len(singles[st]) / n_single) for st in labs}
        alloc[labs[0]] += need - sum(alloc.values())       # rounding
        for st in labs:
            free = [u for u in singles[st] if u not in where]
            for u in free[:alloc[st]]:
                put(u, h)
    for u in units:
        if u not in where:
            put(u, "train")
    part = {k: sorted(i for u, h in where.items() if h == k for i in units[u])
            for k in ("train", "val", "test")}
    return part, where

PART, WHERE = make_split(ROWS, SEED)
for k in ("train", "val", "test"):
    if len(PART[k]) != SPLIT[k]:
        raise SystemExit(f"{k}: {len(PART[k])} rows, expected {SPLIT[k]}")

# ---- leakage checks: refuse to continue if any unit crosses a split -----------
sets = {k: [ROWS[i] for i in v] for k, v in PART.items()}
bnode = {k: {r["meta"]["node"] for r in v if r["meta"]["blind"]} for k, v in sets.items()}
iset  = {k: {(r["meta"]["node"], r["meta"]["iso_class"]) for r in v if not r["meta"]["blind"]} for k, v in sets.items()}
prm   = {k: {r["prompt"] for r in v} for k, v in sets.items()}
grp   = {k: {r["meta"]["group_id"] for r in v if r["meta"].get("group_id")} for k, v in sets.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    for name, d in (("blind situation", bnode), ("seen infoset", iset), ("prompt", prm), ("contrast group", grp)):
        if d[a] & d[b]:
            raise SystemExit(f"LEAK: {len(d[a] & d[b])} {name}(s) in both {a} and {b}")

def distinct(rs):                      # identical prompts are one question
    return list({r["prompt"]: r for r in rs}.values())

TRAIN     = sets["train"]              # file order: contrast groups stay contiguous
VAL_FULL  = distinct(sets["val"])
TEST_ITEMS = distinct(sets["test"])
vb = [r for r in VAL_FULL if r["meta"]["blind"]]
vs = [r for r in VAL_FULL if not r["meta"]["blind"]]
rng = random.Random(SEED + 1)
per = (VAL_DURING_TRAIN - len(vb)) // 4
VAL_SMALL = vb[:]
for a in ("pack", "chaal", "raise", "show"):
    pool = [r for r in vs if r["correct_action"] == a]
    VAL_SMALL += rng.sample(pool, min(per, len(pool)))

if MODE == "sanity":
    # Overfit test: a SMALL set seen MANY times. 128 items x 40 epochs = 80 optimiser
    # steps -- enough for a working pipeline to memorise them, so failing really means a
    # bug. (500 items x 3 epochs was only 24 steps: too few to memorise, a false alarm.)
    TRAIN = random.Random(SEED).sample(TRAIN, 128)
    VAL_SMALL = VAL_FULL = TRAIN
    EPOCHS, VAL_EVERY, SAVE_EVERY = 40, 20, 10**9
elif MODE == "pilot":
    TRAIN = TRAIN[:10_000]   # the file is already shuffled, so a prefix is a fair sample
    EPOCHS, VAL_EVERY, SAVE_EVERY = 1, 50, 50
elif MODE == "full":
    EPOCHS = 1
else:
    raise SystemExit(f"unknown MODE {MODE!r}")

json.dump({k: [ROWS[i]["id"] for i in v] for k, v in PART.items()},
          open(os.path.join(OUT_DIR, "split_manifest.json"), "w"))
c = lambda rs: dict(collections.Counter(r["correct_action"] for r in rs).most_common())
print(f"rows      train {len(PART['train']):,} | val {len(PART['val']):,} | test {len(PART['test']):,}")
print(f"blind situations  train {len(bnode['train'])} | val {len(bnode['val'])} | test {len(bnode['test'])}")
print(f"contrast groups   train {len(grp['train']):,} | val {len(grp['val'])} | test {len(grp['test'])}")
print(f"distinct items    val {len(VAL_FULL):,} (during training {len(VAL_SMALL):,}) | test {len(TEST_ITEMS):,}")
print(f"train labels {c(TRAIN)}")
print(f"val   labels {c(VAL_FULL)}")
print(f"test  labels {c(TEST_ITEMS)}")
print(f"MODE={MODE}: training on {len(TRAIN):,} rows | leakage checks: PASS | manifest saved")

## 6. Hugging Face login and tokenizer

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(BASE_ID)
PAD_ID = tok.convert_tokens_to_ids("<|finetune_right_pad_id|>")
if PAD_ID is None or PAD_ID == tok.unk_token_id:
    PAD_ID = tok.eos_token_id
tok.pad_token_id = PAD_ID
EOT_ID = tok.convert_tokens_to_ids("<|eot_id|>")

def build_prompt(user_text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False, date_string=DATE_STRING)

p = build_prompt("probe")
assert p.count("<|begin_of_text|>") == 1, "BOS should appear exactly once"
assert DATE_STRING in p, "date_string not applied -- template changed?"

ACTIONS = ("chaal", "pack", "raise", "show", "see", "stay-blind")
FIRST = {a: tok(a, add_special_tokens=False)["input_ids"][0] for a in ACTIONS}
if len(set(FIRST.values())) != len(ACTIONS):
    raise SystemExit(f"Action words share a first token: {FIRST}. The fast validation scorer "
                     "needs them distinct.")
print("first token per action:", {a: tok.convert_ids_to_tokens(i) for a, i in FIRST.items()})

## 7. Tokenise — loss on the answer only

Each example is the chat-templated prompt followed by the answer word and `<|eot_id|>`.
Every prompt token gets label `-100`, so the whole training signal goes into the decision.

In [ ]:
import numpy as np

def encode(r):
    prompt = tok(build_prompt(r["prompt"]), add_special_tokens=False)["input_ids"]
    answer = tok(r["correct_action"], add_special_tokens=False)["input_ids"] + [EOT_ID]
    ids = prompt + answer
    if len(ids) > MAX_LEN:
        raise SystemExit(f"{r['id']} is {len(ids)} tokens > MAX_LEN {MAX_LEN}")
    return {"input_ids": np.array(ids, dtype=np.int32),
            "labels": np.array([-100] * len(prompt) + answer, dtype=np.int32),
            "weight": float(r.get("weight", 1.0)) if USE_WEIGHTS else 1.0}

t0 = time.time()
TRAIN_ENC = [encode(r) for r in TRAIN]
lens = sorted(len(e["input_ids"]) for e in TRAIN_ENC)
print(f"tokenised {len(TRAIN_ENC):,} in {time.time() - t0:.0f}s | tokens per example: "
      f"median {lens[len(lens) // 2]}, max {lens[-1]} | total {sum(lens) / 1e6:.1f}M")
ex = TRAIN_ENC[0]
print("answer tokens of example 0:", tok.decode([t for t in ex["labels"] if t != -100]))

## 8. Load the base model and helpers

In [ ]:
import torch.nn.functional as F
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") else "torch_dtype"

def load_base():
    m = AutoModelForCausalLM.from_pretrained(BASE_ID, device_map={"": 0}, **{DTYPE_KW: torch.bfloat16})
    gc_ = m.generation_config
    gc_.do_sample = False
    for k in ("temperature", "top_p", "top_k"):
        if hasattr(gc_, k):
            setattr(gc_, k, None)
    return m

def inner(m):
    """The CausalLM underneath a PEFT wrapper (LoRA layers are injected in place)."""
    return m.get_base_model() if hasattr(m, "get_base_model") else m

def left_pad(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, L - len(s):] = torch.tensor(s); att[i, L - len(s):] = 1
    return ids, att

@torch.no_grad()
def score_items(m, items, bs=16):
    """Fast validation: which legal action's first token does the model rank highest?"""
    was_training = m.training
    m.eval()
    out = []
    for s in range(0, len(items), bs):
        batch = items[s:s + bs]
        ids, att = left_pad([tok(build_prompt(i["prompt"]), add_special_tokens=False)["input_ids"]
                             for i in batch])
        ids, att = ids.to(m.device), att.to(m.device)
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        base = inner(m)
        hs = base.model(input_ids=ids, attention_mask=att, position_ids=pos,
                        use_cache=False).last_hidden_state[:, -1]
        logits = base.lm_head(hs).float()
        for it, lg in zip(batch, logits):
            out.append(max(it["legal_actions"], key=lambda a: lg[FIRST[a]].item()))
    if was_training:
        m.train()
    return out

def val_report(items, preds):
    """Accuracy, balanced accuracy and the collapse checks that matter here."""
    rec = {}
    for a in ACTIONS:
        sub = [k for k, i in enumerate(items) if i["correct_action"] == a]
        if sub:
            rec[a] = sum(preds[k] == a for k in sub) / len(sub)
    look = [preds[k] for k, i in enumerate(items) if i["meta"]["family"] == "look"]
    bb = [preds[k] for k, i in enumerate(items) if i["meta"]["family"] == "blind_bet"]
    seen = [(i, preds[k]) for k, i in enumerate(items) if i["meta"]["family"] == "seen_bet"]
    fold = lambda lo, hi: (lambda s: sum(p == "pack" for _, p in s) / len(s) if s else float("nan"))(
        [(i, p) for i, p in seen if lo <= (i["meta"]["hand_bucket"] or 0) <= hi])
    return {"acc": sum(p == i["correct_action"] for i, p in zip(items, preds)) / len(items),
            "balanced_acc": sum(rec.values()) / len(rec),
            "recall": rec,
            "look_distinct_answers": len(set(look)),
            "blind_fold_rate": (sum(p == "pack" for p in bb) / len(bb)) if bb else float("nan"),
            "fold_weak": fold(0, 8), "fold_strong": fold(17, 24)}

def fmt(r):
    rc = " ".join(f"{a[:5]} {100 * v:3.0f}" for a, v in r["recall"].items())
    return (f"acc {100 * r['acc']:5.1f}  bal {100 * r['balanced_acc']:5.1f} | recall {rc} | "
            f"look answers {r['look_distinct_answers']} | blind fold {100 * r['blind_fold_rate']:4.0f}% | "
            f"fold weak {100 * r['fold_weak']:3.0f}% strong {100 * r['fold_strong']:3.0f}%")

# ---- generation-based evaluation, identical to the batch eval notebook -------
def extract_answer(completion):
    import re
    text = completion.strip().lower()
    hits = [(mm.start(), w) for w in ("stay-blind", "see", "chaal", "raise", "pack", "show")
            for mm in re.finditer(rf"(?<![a-z-]){re.escape(w)}(?![a-z-])", text)]
    return min(hits)[1] if hits else text

@torch.no_grad()
def generate_all(m, items, desc):
    from tqdm.auto import tqdm
    m.eval()
    tok.padding_side = "left"
    out = []
    for s in tqdm(range(0, len(items), GEN_BATCH), desc=desc):
        batch = items[s:s + GEN_BATCH]
        enc = tok([build_prompt(i["prompt"]) for i in batch], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(m.device)
        g = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                       pad_token_id=PAD_ID, use_cache=True)
        for it, t in zip(batch, tok.batch_decode(g[:, enc["input_ids"].shape[1]:],
                                                 skip_special_tokens=True)):
            out.append({"id": it["id"], "answer": extract_answer(t)})
    return out

TEST_SETS = {"internal_test": (lambda: TEST_ITEMS, "teenpattibench_finetune.jsonl (held-out test split)")}
ACTIVE_TESTS = ["internal_test"]

def test_path(name, label):
    return os.path.join(OUT_DIR, f"preds_{TAG}_{label}_{name}.jsonl")

def write_and_score(m, records, name, label, extra=None):
    items = TEST_SETS[name][0]()
    path = test_path(name, label)
    meta = {"type": "metadata",
            "model": {"name": BASE_NAME + ("" if label == "base_l4" else " + TeenPatti LoRA"),
                      "model_id": BASE_ID, "architecture": list(inner(m).config.architectures),
                      "model_type": inner(m).config.model_type,
                      "num_parameters": int(sum(p.numel() for p in inner(m).parameters())),
                      "dtype": "torch.bfloat16", "device": "cuda:0", "quantized": False},
            "tokenizer": {"name_or_path": tok.name_or_path, "vocab_size": tok.vocab_size,
                          "pad_token": tok.convert_ids_to_tokens(PAD_ID), "eos_token": tok.eos_token,
                          "bos_token": tok.bos_token},
            "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "temperature": None,
                           "top_p": None, "top_k": None, "thinking_enabled": False},
            "dataset": {"name": "TeenPattiBench", "split": name,
                        "evaluation_file": TEST_SETS[name][1],
                        "num_questions": len(items)},
            "prompt": {"system_prompt_file": PATHS["teenpattibench_system_prompt.txt"],
                       "system_prompt": SYSTEM_PROMPT, "date_string": DATE_STRING},
            "runtime": {"pytorch_version": torch.__version__, "cuda_available": True,
                        "cuda_version": torch.version.cuda, "gpu_count": torch.cuda.device_count(),
                        "gpus": [GPU], "transformers_version": transformers.__version__},
            "output": {"format": "JSONL", "prediction_file": path}}
    if extra:
        meta["finetune"] = extra
    with open(path + ".part", "w") as fh:
        fh.write(json.dumps(meta) + "\n")
        for r in records:
            fh.write(json.dumps(r) + "\n")
    os.replace(path + ".part", path)
    rep = metrics.score(items, {r["id"]: {"action": r["answer"]} for r in records})
    stem = path.replace("preds_", "report_").replace(".jsonl", "")
    json.dump(rep, open(stem + ".json", "w"), indent=1)
    open(stem + ".txt", "w").write(metrics.format_report(rep, metrics.trivial_baselines(items)) + "\n")
    return rep

model = load_base()
print(f"base model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params | "
      f"GPU memory {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

## 9. Baseline: base Llama on this GPU

Same prompt, decoding and scorer, on *this* hardware, on the internal test split — so
the fine-tuned result is compared like with like. Skipped if already done.

In [ ]:
BASELINE = {}
if RUN_BASELINE and MODE != "sanity":
    for name in ACTIVE_TESTS:
        items, path = TEST_SETS[name][0](), test_path(name, "base_l4")
        if os.path.exists(path):
            recs = [json.loads(l) for l in open(path)][1:]
            BASELINE[name] = metrics.score(items, {r["id"]: {"action": r["answer"]} for r in recs})
            print(f"{name}: already done")
        else:
            BASELINE[name] = write_and_score(model, generate_all(model, items, f"base {name}"), name, "base_l4")
        o = BASELINE[name]["overall"]
        print(f"  base {name:<19} Action Accuracy {100 * o['aa'] / o['n']:.2f}%  (n={o['n']:,})")
base_val = val_report(VAL_SMALL, score_items(model, VAL_SMALL))
print("\nbase model on validation (training subset):\n  " + fmt(base_val))

## 10. Train

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, TrainerCallback
from torch.utils.data import SequentialSampler

DONE_FLAG = os.path.join(OUT_DIR, "train_done.json")
model.config.use_cache = False
if GRAD_CHECKPOINTING:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.from_numpy(b["input_ids"].astype(np.int64))
        lab[i, :n] = torch.from_numpy(b["labels"].astype(np.int64))
        att[i, :n] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab,
            "weight": torch.tensor([b["weight"] for b in batch], dtype=torch.float32)}

class TPTrainer(Trainer):
    """Answer-only, optionally weighted loss; file order kept so contrast groups stay together."""
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.model_accepts_loss_kwargs = False   # always average over accumulation steps

    def _get_train_sampler(self, *a, **k):
        return SequentialSampler(self.train_dataset)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        base = inner(model)
        labels, w = inputs["labels"], inputs["weight"]
        hs = base.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                        use_cache=False).last_hidden_state
        sel = labels[:, 1:] != -100                  # positions whose NEXT token is answer
        rows = sel.nonzero(as_tuple=True)[0]
        logits = base.lm_head(hs[:, :-1][sel]).float()   # only ~2-4 positions per example
        ce = F.cross_entropy(logits, labels[:, 1:][sel], reduction="none")
        B = labels.shape[0]
        per_ex = torch.zeros(B, device=ce.device).index_add_(0, rows, ce) / \
                 torch.zeros(B, device=ce.device).index_add_(0, rows, torch.ones_like(ce)).clamp(min=1)
        loss = (per_ex * w).sum() / w.sum()
        return (loss, None) if return_outputs else loss

VAL_HISTORY = [json.loads(l) for l in open(VAL_LOG)] if os.path.exists(VAL_LOG) else []

class ValCallback(TrainerCallback):
    def __init__(self):
        self.best = -1.0
        if os.path.exists(os.path.join(BEST_DIR, "best.json")):
            self.best = json.load(open(os.path.join(BEST_DIR, "best.json")))["balanced_acc"]
    def _run(self, state):
        t = time.time()
        r = val_report(VAL_SMALL, score_items(model, VAL_SMALL))
        r["step"] = state.global_step
        VAL_HISTORY.append(r)
        with open(VAL_LOG, "a") as fh:
            fh.write(json.dumps(r) + "\n")
        flag = ""
        if r["balanced_acc"] > self.best:
            self.best = r["balanced_acc"]
            model.save_pretrained(BEST_DIR)
            json.dump(r, open(os.path.join(BEST_DIR, "best.json"), "w"), indent=1)
            shutil.copy(os.path.join(OUT_DIR, "run_config.json"), os.path.join(BEST_DIR, "run_config.json"))
            flag = "  <- best, saved"
        print(f"\n[val step {state.global_step}] {fmt(r)}{flag}  ({time.time() - t:.0f}s)")
    def on_step_end(self, args, state, control, **k):
        if state.global_step % VAL_EVERY == 0:
            self._run(state)
    def on_train_end(self, args, state, control, **k):
        if state.global_step % VAL_EVERY:
            self._run(state)


TRAIN_LOG = os.path.join(OUT_DIR, "train_log.jsonl")

class LiveMonitor(TrainerCallback):
    # One dashboard, updated in place: status line + loss + validation curves.
    # Complements the trainer's own progress bar and loss table rather than replacing
    # them. The figure is also saved as training_progress.png in the run folder.
    def __init__(self, total_steps):
        self.total = total_steps
        self.loss = [json.loads(l) for l in open(TRAIN_LOG)] if os.path.exists(TRAIN_LOG) else []
        self.t0, self.s0, self.handle = None, None, None
    def on_log(self, args, state, control, logs=None, **k):
        if logs and "loss" in logs:
            rec = {"step": state.global_step, "loss": logs["loss"],
                   "lr": logs.get("learning_rate"), "time": time.time()}
            self.loss.append(rec)
            with open(TRAIN_LOG, "a") as fh:
                fh.write(json.dumps(rec) + "\n")
    def on_step_end(self, args, state, control, **k):
        if self.t0 is None:
            self.t0, self.s0 = time.time(), state.global_step
        if state.global_step % PLOT_EVERY == 0 or state.global_step == self.total:
            self.draw(state.global_step)
    def on_train_end(self, args, state, control, **k):
        self.draw(state.global_step)
    def draw(self, step):
        try:
            self._draw(step)
        except Exception as e:                  # a plotting problem must never stop training
            print(f"(dashboard skipped at step {step}: {type(e).__name__}: {e})")
    def _draw(self, step):
        import io
        import matplotlib.pyplot as plt
        from IPython.display import display, Image
        done = step / self.total
        rate = (step - self.s0) / max(1e-9, time.time() - self.t0)       # steps / s, this session
        left = (self.total - step) / rate if rate > 0 else float("nan")
        hms = lambda x: "--" if x != x else f"{int(x // 3600)}h{int(x % 3600 // 60):02d}m"
        last = self.loss[-1] if self.loss else {}
        status = (f"step {step:,}/{self.total:,} ({100 * done:.1f}%)  |  "
                  f"{hms(time.time() - self.t0)} elapsed, ~{hms(left)} left  |  "
                  f"{step * PER_DEVICE_BS * GRAD_ACCUM:,} examples  |  "
                  f"loss {last.get('loss', float('nan')):.3f}  |  lr {last.get('lr') or 0:.1e}  |  "
                  f"GPU {torch.cuda.max_memory_allocated(0) / 1e9:.1f} GB peak")
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
        fig.suptitle(status, fontsize=10, x=0.01, ha="left")
        if self.loss:
            xs = [r["step"] for r in self.loss]; ys = [r["loss"] for r in self.loss]
            ax[0].plot(xs, ys, alpha=0.35, lw=1, label="loss (every 10 steps)")
            w = 10
            if len(ys) >= w:
                ax[0].plot(xs[w - 1:], [sum(ys[i - w + 1:i + 1]) / w for i in range(w - 1, len(ys))],
                           lw=2, label="moving average")
        ax[0].set_title("training loss (answer tokens)"); ax[0].set_xlabel("step")
        ax[0].set_xlim(0, self.total); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
        if VAL_HISTORY:
            vx = [r["step"] for r in VAL_HISTORY]
            ax[1].plot(vx, [100 * r["acc"] for r in VAL_HISTORY], "o-", label="accuracy")
            ax[1].plot(vx, [100 * r["balanced_acc"] for r in VAL_HISTORY], "s-", label="balanced accuracy")
            for a in ACTIONS:
                ys = [100 * r["recall"].get(a, float("nan")) for r in VAL_HISTORY]
                ax[2].plot(vx, ys, "o-", ms=3, label=a)
        if "base_val" in globals():
            ax[1].axhline(100 * base_val["balanced_acc"], ls="--", c="grey", lw=1, label="base model (balanced)")
        ax[1].set_title("validation"); ax[1].set_ylim(0, 100); ax[1].set_xlim(0, self.total)
        ax[1].set_xlabel("step"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
        ax[2].set_title("validation recall per action (0 = never produced)")
        ax[2].set_ylim(-3, 103); ax[2].set_xlim(0, self.total); ax[2].set_xlabel("step")
        if VAL_HISTORY:
            ax[2].legend(fontsize=7, ncol=2)
        else:
            ax[2].text(0.5, 0.5, f"first validation at step {VAL_EVERY}", ha="center",
                       va="center", transform=ax[2].transAxes, color="grey")
        ax[2].grid(alpha=0.3)
        fig.tight_layout(rect=(0, 0, 1, 0.93))
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110)
        plt.close(fig)                          # never left open: no duplicate inline copies
        png = buf.getvalue()
        open(os.path.join(OUT_DIR, "training_progress.png"), "wb").write(png)
        if self.handle is None:                 # one output area, replaced in place
            self.handle = display(Image(data=png), display_id=True)
        else:
            self.handle.update(Image(data=png))

steps = math.ceil(len(TRAIN_ENC) / (PER_DEVICE_BS * GRAD_ACCUM)) * EPOCHS
args = TrainingArguments(
    output_dir=CKPT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_steps=max(1, round(WARMUP_FRAC * steps)), weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM, bf16=True, optim="adamw_torch_fused", logging_steps=10,
    save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=3, report_to="none",
    remove_unused_columns=False, label_names=["labels"], dataloader_num_workers=2,
    gradient_checkpointing=GRAD_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False} if GRAD_CHECKPOINTING else None,
    seed=SEED)

# ---- one complete, human-readable record of this run -------------------------
import hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
RUN_CONFIG = {
    "mode": MODE, "base_model": BASE_ID, "seed": SEED,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "optimisation": {"lr": LR, "schedule": "cosine", "warmup_frac": WARMUP_FRAC,
                     "warmup_steps": args.warmup_steps, "weight_decay": WEIGHT_DECAY,
                     "max_grad_norm": MAX_GRAD_NORM, "optimizer": args.optim,
                     "per_device_batch": PER_DEVICE_BS, "grad_accum": GRAD_ACCUM,
                     "effective_batch": PER_DEVICE_BS * GRAD_ACCUM, "epochs": EPOCHS,
                     "optimiser_steps": steps, "precision": "bf16",
                     "gradient_checkpointing": GRAD_CHECKPOINTING, "packing": False,
                     "sampler": "sequential (keeps contrast groups together)"},
    "loss": {"answer_tokens_only": True, "target": "action word + <|eot_id|>",
             "per_example_weight": USE_WEIGHTS, "max_len": MAX_LEN},
    "data": {"finetune_file": PATHS["teenpattibench_finetune.jsonl"],
             "finetune_sha256": sha256(PATHS["teenpattibench_finetune.jsonl"]),
             "system_prompt_sha256": sha256(PATHS["teenpattibench_system_prompt.txt"]),
             "split_rows": SPLIT, "train_rows_used": len(TRAIN_ENC),
             "val_distinct": len(VAL_FULL), "val_during_training": len(VAL_SMALL),
             "test_distinct": len(TEST_ITEMS), "benchmark_eval_used": False,
             "split_manifest": "split_manifest.json"},
    "prompt": {"chat_template": "tokenizer.apply_chat_template, system role",
               "date_string": DATE_STRING, "thinking": False},
    "validation": {"every_steps": VAL_EVERY, "select_by": "balanced accuracy (mean recall over 6 actions)",
                   "scorer": "first token of each legal action"},
    "checkpoints": {"every_steps": SAVE_EVERY, "keep_last": 3},
    "evaluation": {"decoding": "greedy", "max_new_tokens": MAX_NEW_TOKENS, "batch": GEN_BATCH},
    "runtime": {"gpu": GPU, "torch": torch.__version__, "cuda": torch.version.cuda,
                "transformers": transformers.__version__, "peft": peft.__version__,
                "python": sys.version.split()[0]},
    "trainer_args": json.loads(args.to_json_string()),
}
json.dump(RUN_CONFIG, open(os.path.join(OUT_DIR, "run_config.json"), "w"), indent=1)
print(f"saved run_config.json ({len(json.dumps(RUN_CONFIG)):,} chars) -- every setting of this run")

if os.path.exists(DONE_FLAG):
    print("training already finished in an earlier session -- skipping to evaluation")
else:
    trainer = TPTrainer(model=model, args=args, train_dataset=TRAIN_ENC, data_collator=collate,
                        callbacks=[ValCallback(), LiveMonitor(steps)])
    resume = os.path.isdir(CKPT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(CKPT_DIR))
    print(f"{len(TRAIN_ENC):,} examples x {EPOCHS} epoch(s) = {steps:,} optimiser steps"
          + (" -- RESUMING from the last checkpoint" if resume else ""))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=True if resume else None)
    json.dump({"minutes": round((time.time() - t0) / 60, 1), "steps": steps, "mode": MODE,
               "train_items": len(TRAIN_ENC)}, open(DONE_FLAG, "w"))
    print(f"training finished in {(time.time() - t0) / 60:.1f} min")

## 11. Best adapter on the full validation set

In [ ]:
from peft import PeftModel
best = json.load(open(os.path.join(BEST_DIR, "best.json")))
print(f"best adapter: step {best['step']}, small-val balanced accuracy {100 * best['balanced_acc']:.1f}%")
del model; gc.collect(); torch.cuda.empty_cache()
model = PeftModel.from_pretrained(load_base(), BEST_DIR)
model.eval()
full_val = val_report(VAL_FULL, score_items(model, VAL_FULL))
print("full validation:\n  " + fmt(full_val))
if MODE == "sanity":
    gain = full_val["acc"] - base_val["acc"]
    print(f"\nbase model {100 * base_val['acc']:.1f}% -> after training {100 * full_val['acc']:.1f}% "
          f"on the same {len(VAL_FULL)} items")
    if full_val["acc"] >= 0.9:
        print("SANITY PASS -- it memorised what it was shown. Set MODE = 'pilot' and Run all.")
    elif gain >= 0.15:
        print("SANITY PARTIAL -- clearly learning (+%.0f points) but not memorised. The pipeline "
              "works; memorising needs more steps. Safe to go to 'pilot'." % (100 * gain))
    else:
        print("SANITY FAIL -- barely moved from the base model. Something is wrong with the "
              "loss or the labels. Stop and send the output.")

## 12. Test once: internal test split

Generation, not the fast scorer: exactly the procedure used for every model in the
benchmark. Written in the standard prediction format, plus a `finetune` block recording
how the adapter was trained.

In [ ]:
FINAL = {}
if RUN_FINAL_TEST and MODE != "sanity":
    extra = {"method": "LoRA", "r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
             "targets": LORA_TARGETS, "lr": LR, "epochs": EPOCHS, "effective_batch": PER_DEVICE_BS * GRAD_ACCUM,
             "split": SPLIT, "train_items": len(TRAIN_ENC), "mode": MODE, "best_step": best["step"],
             "val_balanced_acc": best["balanced_acc"], "use_weights": USE_WEIGHTS, "seed": SEED}
    for name in ACTIVE_TESTS:
        FINAL[name] = write_and_score(model, generate_all(model, TEST_SETS[name][0](), f"fine-tuned {name}"),
                                      name, f"ft_{MODE}_l4", extra)

    def strata(rep):
        f = lambda k: (100 * rep["by_family"][k]["aa"] / rep["by_family"][k]["n"]) if k in rep["by_family"] else float("nan")
        return (100 * rep["overall"]["aa"] / rep["overall"]["n"], f("seen_bet"), f("look"), f("blind_bet"))
    for name in ACTIVE_TESTS:
        items = TEST_SETS[name][0]()
        g = [i for i in items if not i["is_mixed"]]
        base_ = {}
        for fam, sub in (("all", g), ("seen", [i for i in g if i["meta"]["family"] == "seen_bet"]),
                         ("conv", [i for i in g if i["meta"]["family"] == "look"]),
                         ("blind", [i for i in g if i["meta"]["family"] == "blind_bet"])):
            base_[fam] = max((100 * sum(i["correct_action"] == a for i in sub) / len(sub))
                             for a in ACTIONS) if sub else float("nan")
        print(f"\n=== {name}  ({len(g):,} graded) ===")
        print(f"{'':<20}{'all':>8}{'seen':>8}{'conv':>8}{'blind':>8}")
        print(f"{'one word forever':<20}" + "".join(f"{base_[k]:8.2f}" for k in ("all", "seen", "conv", "blind")))
        if name in BASELINE:
            print(f"{'base':<20}" + "".join(f"{x:8.2f}" for x in strata(BASELINE[name])))
        print(f"{'fine-tuned':<20}" + "".join(f"{x:8.2f}" for x in strata(FINAL[name])))

        recs = [json.loads(l) for l in open(test_path(name, f"ft_{MODE}_l4"))][1:]
        ans = {r["id"]: r["answer"] for r in recs}
        look = [i for i in g if i["meta"]["family"] == "look"]
        if look:
            print(f"conversion answers: {dict(collections.Counter(ans[i['id']] for i in look))}"
                  "   (a single answer for every item = no timing learned)")
            for t in (1, 2, 3, 4):
                s_ = [i for i in look if i["meta"]["turn_index"] == t]
                if s_:
                    print(f"  turn {t}: {100 * sum(ans[i['id']] == i['correct_action'] for i in s_) / len(s_):5.1f}%  (n={len(s_)})")
    print("\nThis is the internal test, drawn from the same generator as training. It is not"
          "\ncomparable to the nine benchmark models; evaluate the adapter on the benchmark"
          "\neval in a separate run for that.")

## 13. Download everything

In [ ]:
import zipfile
z = f"/content/{TAG}_{MODE}_finetune_results.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(OUT_DIR):
        if "checkpoints" in root:
            continue                                  # large; the best adapter is kept
        for f in fs:
            if not f.endswith(".part"):
                full = os.path.join(root, f)
                zf.write(full, os.path.relpath(full, OUT_DIR))
print(f"{z}: {os.path.getsize(z) / 1e6:.0f} MB")
from google.colab import files
files.download(z)